# Google Landmark Detection — Exploratory Data Analysis

This notebook walks through dataset loading, EDA, and a quick
training sanity-check.  Run cells top-to-bottom after placing
`train.csv` and images under `data/raw/`.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

from src.config import get_config
from src.dataset import load_metadata, exploratory_data_analysis, filter_top_k_classes, encode_labels, add_image_paths, stratified_split
from src.visualization import plot_sample_images

config = get_config()
config.ensure_directories()
print('Project root:', config.root)

## 1. Load metadata

In [ ]:
df = load_metadata(config.paths.train_csv)
df.head()

## 2. Exploratory Data Analysis

In [ ]:
stats = exploratory_data_analysis(
    df,
    base_dir=config.paths.data_raw,
    save_dir=config.paths.figures,
)
stats['total_images'], stats['unique_landmarks'], stats['imbalance_ratio']

## 3. Filter to top-K classes

In [ ]:
filtered = filter_top_k_classes(
    df,
    top_k=config.dataset.top_k_classes,
    min_samples=config.dataset.min_samples_per_class,
    max_samples=config.dataset.max_samples_per_class,
)
filtered['landmark_id'].value_counts().describe()

## 4. Encode labels & add paths

In [ ]:
filtered, encoder = encode_labels(filtered, encoder_path=config.paths.encoder_file)
filtered = add_image_paths(filtered, base_dir=config.paths.data_raw)
filtered.head()

## 5. Visualise sample images

In [ ]:
plot_sample_images(filtered, base_dir=config.paths.data_raw, n=16, config=config)

## 6. Stratified split

In [ ]:
train_df, val_df, test_df = stratified_split(filtered, val_size=config.dataset.val_split, test_size=config.dataset.test_split, seed=config.seed)
len(train_df), len(val_df), len(test_df)

## 7. Train (run from terminal for full training)

```bash
python -m src.train
```